## Process Lab Result Data

In [ ]:
import os

import pandas as pd
import numpy as np

from dateutil.relativedelta import relativedelta
from math import sqrt, log, e

data_path = os.path.join('..', 'data/')
raw_data_path = os.path.join(data_path, 'raw_data')
processed_data_path = os.path.join(data_path, 'processed_data')

## Load Data

### Patients of Interest

In [ ]:
cols = ['master_person_id', 'inclusion_date', 'endpoint_date', 'dateOfBirth', 'gender']

inclusion_patients_df = pd.read_csv(os.path.join(data_path, "opt_in_ckd_inclusion_patients.csv"))[cols]

del cols

inclusion_patients_df['inclusion_date'] = pd.to_datetime(inclusion_patients_df['inclusion_date']).dt.date
inclusion_patients_df['endpoint_date'] = pd.to_datetime(inclusion_patients_df['endpoint_date']).dt.date
inclusion_patients_df['dateOfBirth'] = pd.to_datetime(inclusion_patients_df['dateOfBirth']).dt.date

### Lab Results Data

In [ ]:
lab_results_df = pd.read_csv(os.path.join(raw_data_path, "20251118_lab_search_results.csv"))

lab_results_df['measurement_date'] = pd.to_datetime(lab_results_df['measurement_date']).dt.date

### Date Breakdown Data

In [ ]:
dates_breakdown_df = pd.read_csv(os.path.join(data_path, "ckd_patients_datebreakdown.csv"))

dates_breakdown_df['inclusion_date'] = pd.to_datetime(dates_breakdown_df['inclusion_date']).dt.date
dates_breakdown_df['endpoint_date'] = pd.to_datetime(dates_breakdown_df['endpoint_date']).dt.date
dates_breakdown_df['start_date'] = pd.to_datetime(dates_breakdown_df['start_date']).dt.date
dates_breakdown_df['end_date'] = pd.to_datetime(dates_breakdown_df['end_date']).dt.date

## Refined Lab Results

In [ ]:
lab_results_refined_df = inclusion_patients_df.merge(lab_results_df, how='inner', on='master_person_id')

del inclusion_patients_df, lab_results_df

inclusion_date_filter = (lab_results_refined_df['measurement_date']>=lab_results_refined_df['inclusion_date'])
endpoint_date_filter = (lab_results_refined_df['measurement_date']<=lab_results_refined_df['endpoint_date'])

cols = ['master_person_id', 'dateOfBirth', 'gender', 'inclusion_date', 'endpoint_date', 'measurement_date', 'measurement_cleaned', 'measurement_value', 'measurement_unit_cleaned']
rename_dict = {'measurement_cleaned': 'measurement', 'measurement_unit_cleaned': 'measurement_unit'}

lab_results_refined_df = lab_results_refined_df[inclusion_date_filter&endpoint_date_filter][cols].reset_index(drop=True).rename(columns=rename_dict)

del inclusion_date_filter, endpoint_date_filter, cols, rename_dict

lab_results_refined_df.head()

### Standardise Measures

In [ ]:
def measure_standard(row):
    if row['measurement']=='Haemoglobin' and row['measurement_unit']!='g/L':
        return row['measurement_value']*10
    elif row['measurement']=='HbA1c' and row['measurement_unit']=='%':
        return 10.929*(row['measurement_value']-2.15)
    elif row['measurement']=='NT-proBNP' and row['measurement_unit']=='pmol/L':
        return row['measurement_value']/2.247
    elif row['measurement']=='Troponin I' and row['measurement_unit']=='ng/mL':
        return row['measurement_value']*1000
    elif row['measurement']=='Creatinine' and row['measurement_unit']=='mmol/L':
        return row['measurement_value']*1000
    elif row['measurement']=='Urine Creatinine' and row['measurement_unit']=='µmol/L':
        return row['measurement_value']/1000
    elif row['measurement']=='Urine Protein' and row['measurement_unit']=='mg/L':
        return row['measurement_value']/1000
    elif row['measurement']=='Random Glucose' and row['measurement_unit']=='mg/dL':
        return row['measurement_value']/18.0182
    elif row['measurement']=='Urine PCR' and row['measurement_unit']=='g/mmol':
        return row['measurement_value']/1000
    elif row['measurement']=='Urine ACR' and row['measurement_unit']=='mg/mol':
        return row['measurement_value']*1000
    else:
        return row['measurement_value']

In [ ]:
lab_results_refined_df['measurement_value'] = lab_results_refined_df.apply(lambda row: measure_standard(row), axis=1)

In [ ]:
def unit_standard(row):
    if row['measurement']=='Haemoglobin':
        return 'g/L'
    elif row['measurement']=='HbA1c':
        return 'mmol/mol'
    elif row['measurement']=='NT-proBNP':
        return 'ng/L'
    elif row['measurement']=='Troponin I':
        return 'ng/L'
    elif row['measurement']=='Creatinine':
        return 'µmol/L'
    elif row['measurement']=='Urine Creatinine':
        return 'mmol/L'
    elif row['measurement']=='Urine Protein':
        return 'g/L'
    elif row['measurement']=='eGFR':
        return 'mL/min'
    elif row['measurement']=='Random Glucose':
        return 'mmol/L'
    elif row['measurement']=='Cholesterol':
        return 'mmol/L'
    elif row['measurement']=='Urine PCR':
        return 'mg/mmol'
    elif row['measurement']=='Urine ACR':
        return 'g/mmol'
    else:
        return row['measurement_unit']

In [ ]:
lab_results_refined_df['measurement_unit'] = lab_results_refined_df.apply(lambda row: unit_standard(row), axis=1)

In [ ]:
lab_results_refined_df['measurement'] = lab_results_refined_df['measurement'].str.replace(' ', '_').str.replace('-', '_')

## Pivot Table
- Aim to create One Row per measurement date

In [ ]:
def gfr_calculated(row):
    if not pd.isnull(row['Creatinine']) and row['Creatinine']!=0:
        src = row['Creatinine']/88.42
        age_calc = 0.9938**row['age_at_measurement_date']
        
        if row['gender'] == 'Female':
            K = 0.7
            a = -0.241
            min_src = min(src/K, 1)**a
            max_src = max(src/K, 1)**-1.209
            multiplication = 1.012
        elif row['gender'] == 'Male':
            K = 0.9
            a = -0.302
            min_src = min(src/K, 1)**a
            max_src = max(src/K, 1)**-1.209
            multiplication = 1

        return round(142*min_src*max_src*age_calc*multiplication, 2)

In [ ]:
def urine_acr_calculation(row):
    creatinine = row['Urine_Creatinine']
    albumin = row['Urine_Albumin']

    if not pd.isnull(creatinine) and creatinine != 0 and not pd.isnull(albumin) and pd.isnull(row['Urine_ACR']):
        creatinine *= 11.312
        albumin *= 0.1
    
        return (albumin/creatinine)*0.113
    else:
        return row['Urine_ACR']

In [ ]:
def urine_pcr_calculation(row):
    creatinine = row['Urine_Creatinine']
    protein = row['Urine_Protein']

    if not pd.isnull(creatinine) and creatinine != 0 and not pd.isnull(protein) and pd.isnull(row['Urine_PCR']):
        creatinine *= 11.312
        protein *= 100
    
        return (protein/creatinine)*0.113
    else:
        return row['Urine_PCR']

In [ ]:
def kfre_calculation(row):
    if not pd.isnull(row['Urine_ACR']):
        acr = row['Urine_ACR']/1000
    elif not pd.isnull(row['Urine_PCR']):
        acr = row['Urine_PCR']*0.7
    else:
        acr = np.NAN
    
    if not pd.isnull(acr) and not pd.isnull(row['gfr_calculated']) and acr!=0:
        age = row['age_at_measurement_date']
        gfr = row['gfr_calculated']

        if row['gender'] == 'Female':
            sex = 0
        else:
            sex = 1

        return (1-0.957**e**((-0.2201*(age/10-7.036))+(0.2467*(sex-0.5642))-(0.5567*(gfr/5-7.222))+(0.451*(log(acr/0.113)-5.137))))*100

In [ ]:
lab_results_pivot_df = lab_results_refined_df.pivot_table(columns='measurement',
                                                          values=['measurement_value'],
                                                          index=['master_person_id', 'gender', 'dateOfBirth', 'measurement_date'],
                                                          aggfunc='first'
                                                          ).reset_index()


cols = list(lab_results_pivot_df.columns.get_level_values(0)[:4])
cols.extend(list(lab_results_pivot_df.columns.get_level_values(1)[4:]))

lab_results_pivot_df.columns = cols

del lab_results_refined_df, cols

lab_results_pivot_df.insert(3, 'age_at_measurement_date', lab_results_pivot_df.apply(lambda row: relativedelta(row['measurement_date'], row['dateOfBirth']).years, axis=1))
lab_results_pivot_df['gfr_calculated'] = lab_results_pivot_df.apply(lambda row: gfr_calculated(row), axis=1)
lab_results_pivot_df['Urine_ACR'] = lab_results_pivot_df.apply(lambda row: urine_acr_calculation(row), axis=1)
lab_results_pivot_df['Urine_PCR'] = lab_results_pivot_df.apply(lambda row: urine_pcr_calculation(row), axis=1)

lab_results_pivot_df['kfre'] = lab_results_pivot_df.apply(lambda row: kfre_calculation(row), axis=1)

lab_results_pivot_df = lab_results_pivot_df.drop(columns=['dateOfBirth', 'gender', 'age_at_measurement_date'])

lab_results_pivot_df.head()

## Group Measurements to Date Breakdown

In [ ]:
drop_cols = ['inclusion_date', 'endpoint_date']

lab_results_df = dates_breakdown_df.merge(lab_results_pivot_df, how='right', on='master_person_id')

del lab_results_pivot_df

start_date_filter = (lab_results_df['measurement_date']>=lab_results_df['start_date'])
end_date_filter = (lab_results_df['measurement_date']<=lab_results_df['end_date'])

lab_results_df = lab_results_df[start_date_filter&end_date_filter].drop_duplicates().drop(columns=drop_cols).reset_index(drop=True)

del drop_cols, start_date_filter, end_date_filter

lab_results_df['grouping'] = lab_results_df['grouping'].astype(int)

In [ ]:
group_cols = list(lab_results_df.columns[:2])
agg = {}

for col in lab_results_df.columns[5:]:
     agg[col] = 'mean'

aggregated_lab_results_df = lab_results_df.groupby(group_cols).aggregate(agg).reset_index()

del lab_results_df, group_cols, agg

In [ ]:
full_lab_results_df = dates_breakdown_df.merge(aggregated_lab_results_df, how='left', on=['master_person_id', 'grouping'])

del dates_breakdown_df, aggregated_lab_results_df

for col in full_lab_results_df.columns[6:]:
    full_lab_results_df[col] = full_lab_results_df[col].apply(lambda x: round(x, 2))

full_lab_results_df.head()

### Calculate GFR Slope

In [ ]:
def years_difference(row):
    relative_delta = relativedelta(row['start_date_shifted'], row['start_date'])
    years_diff = int(abs(relative_delta.years + relative_delta.months/12 + relative_delta.days/365))

    return years_diff

cols = ['master_person_id', 'start_date_shifted', 'start_date', 'eGFR_shifted', 'eGFR', 'gfr_calculated_shifted', 'gfr_calculated']

gfr_df = full_lab_results_df[full_lab_results_df['eGFR'].notna()][['master_person_id', 'start_date', 'eGFR', 'gfr_calculated']]

shifted_gfr_df = gfr_df.shift(1)
shifted_gfr_df.columns = ['master_person_id_shifted', 'start_date_shifted', 'eGFR_shifted', 'gfr_calculated_shifted']

gfr_slope_df = gfr_df.join(shifted_gfr_df)
gfr_slope_df = gfr_slope_df[gfr_slope_df['master_person_id']==gfr_slope_df['master_person_id_shifted']]
gfr_slope_df = gfr_slope_df[cols]

gfr_slope_df.insert(3, 'years_difference', gfr_slope_df.apply(lambda row: years_difference(row), axis=1))

del gfr_df, shifted_gfr_df, years_difference

gfr_slope_df.insert(4, 'eGFR_Slope', (gfr_slope_df['eGFR']-gfr_slope_df['eGFR_shifted'])/gfr_slope_df['years_difference'])
gfr_slope_df.insert(7, 'GFR_Calculated_Slope', (gfr_slope_df['gfr_calculated']-gfr_slope_df['gfr_calculated_shifted'])/gfr_slope_df['years_difference'])

cols.append('years_difference')

gfr_slope_df.drop(columns=cols, inplace=True)

del cols

In [ ]:
full_lab_results_df = full_lab_results_df.join(gfr_slope_df)

del gfr_slope_df

full_lab_results_df['eGFR_Slope'] = full_lab_results_df.apply(lambda row: np.NaN if row['grouping']==1 else row['eGFR_Slope'], axis=1)
full_lab_results_df['GFR_Calculated_Slope'] = full_lab_results_df.apply(lambda row: np.NaN if row['grouping']==1 else row['GFR_Calculated_Slope'], axis=1)

full_lab_results_df.head()

## Export Lab Results Data

In [ ]:
# --- Save Results ---
file_name = "20260306_processed_lab_results.csv"

full_lab_results_df.to_csv(f"{processed_data_path}/{file_name}", index=False)
print("✅ Results saved.")